<!-- course-header -->
# Lesson 07: Text as numbers, sequences and embeddings

*Companion notebook for lesson 07. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/07-sequences-embeddings.html) (it works well on a phone) or [offline](../lessons/07-sequences-embeddings.html). Worked solutions are in `solutions/07-sequences-embeddings-solutions.ipynb`; try each exercise first.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/07-sequences-embeddings.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

How text becomes vectors whose geometry captures meaning, how recurrent networks read sequences one step at a time, and the bottleneck that motivated attention.

**Runs on:** Any computer (CPU). The last cells use PyTorch.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy torch

## The distributional hypothesis

### Embeddings from counting, in twenty lines

Before neural networks, you could build word vectors by counting co-occurrences and compressing the counts with the SVD from lesson 02. Doing it yourself makes the idea concrete.

In [ ]:
import numpy as np

corpus = """the cat sat on the mat . the dog sat on the rug . a cat chased a mouse .
a dog chased a cat . the mouse ate the cheese . the cat ate the fish .
the dog ate the bone . a kitten is a young cat . a puppy is a young dog .
the kitten sat on the rug . the puppy sat on the mat . the mouse ran from the cat .
the dog ran in the park . the cat slept on the mat . the dog slept on the rug .
paris is the capital of france . berlin is the capital of germany . rome is the capital of italy .
france exports wine . germany exports cars . italy exports wine .""".split()

vocab = sorted(set(corpus))
idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

# 1. count how often each word appears within 2 positions of each other word
C = np.zeros((V, V))
for i, w in enumerate(corpus):
    for j in range(max(0, i - 2), min(len(corpus), i + 3)):
        if i != j:
            C[idx[w], idx[corpus[j]]] += 1

# 2. positive pointwise mutual information: how much more often than chance do they co-occur?
total = C.sum()
pw = C.sum(1, keepdims=True) / total
pc = C.sum(0, keepdims=True) / total
with np.errstate(divide="ignore"):
    ppmi = np.maximum(np.log((C / total) / (pw * pc)), 0)

# 3. compress with the SVD: keep 10 dimensions, normalise for cosine similarity
U, S, _ = np.linalg.svd(ppmi)
E = U[:, :10] * S[:10]
E /= np.linalg.norm(E, axis=1, keepdims=True)

def nearest(word, k=3):
    sims = E @ E[idx[word]]
    return [vocab[i] for i in np.argsort(-sims)[1 : k + 1]]

print(nearest("cat"))      # ['mouse', 'dog', 'bone']
print(nearest("mat"))      # ['slept', 'rug', 'sat']
print(nearest("france"))   # ['germany', '.', 'of']: noisy, but germany is first

### Embeddings in PyTorch

An embedding layer is just a lookup table: a matrix of shape `(vocab_size, dim)` whose rows are trained by backprop along with the rest of the model. Indexing it with token ids is equivalent to multiplying one-hot vectors by the matrix, without wasting work on the zeros.

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
import torch, torch.nn as nn

emb = nn.Embedding(num_embeddings=50_000, embedding_dim=256)
token_ids = torch.tensor([[12, 873, 4, 99], [5, 5, 31, 0]])   # (batch=2, seq_len=4)
vectors = emb(token_ids)                                       # (2, 4, 256)

> **Static versus contextual embeddings**
>
> word2vec gives "bank" one vector, whether it is a river bank or a savings bank. Transformers produce **contextual** embeddings: the vector for "bank" changes depending on the surrounding words. That is a large part of why they understand language so much better.

## Recurrent neural networks

The same weights are used at every step, so the network handles any length.

In [ ]:
rng = np.random.default_rng(0)
d_in, d_h = 8, 16
Wx = rng.normal(0, 0.3, (d_in, d_h))
Wh = rng.normal(0, 0.3, (d_h, d_h))
b = np.zeros(d_h)

xs = rng.normal(size=(5, d_in))        # a sequence of 5 input vectors (e.g. word embeddings)
h = np.zeros(d_h)                      # start with an empty memory
for x in xs:
    h = np.tanh(x @ Wx + h @ Wh + b)   # update the memory with each new input
print(h.shape)                         # (16,): a summary of the whole sequence

### Why plain RNNs forget

Training an RNN backpropagates through every time step, multiplying by $W_h$ (and the tanh derivative) again and again. Repeated multiplication either shrinks or explodes:

In [ ]:
for factor in [0.9, 1.0, 1.1]:
    print(factor, factor ** 50)       # 0.0052, 1.0, 117.4

### LSTMs and GRUs

The **LSTM** (long short-term memory) adds a separate cell state that flows through time with only gentle, gated modifications, and three learned **gates** decide what to forget, what to write and what to output. Because the cell state is updated additively, gradients can flow across many more steps (the same insight as residual connections). The **GRU** is a simpler variant with two gates. From roughly 2014 to 2017, LSTMs were the state of the art for translation, speech recognition and language modelling.

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
lstm = nn.LSTM(input_size=256, hidden_size=512, num_layers=2, batch_first=True)
out, (h_n, c_n) = lstm(vectors)        # vectors: (2, 4, 256) from the embedding above
print(out.shape)                       # (2, 4, 512): a hidden state for every position

## Sequence-to-sequence and the bottleneck

### Hands-on: explore real embeddings

Install `gensim` and load small pretrained GloVe vectors (`gensim.downloader.load("glove-wiki-gigaword-50")`, about 66 MB).

1. Find the nearest neighbours of "python", "spectrometer" and "transformer". What do the results tell you about the training corpus?
2. Test analogies with `most_similar(positive=["king", "woman"], negative=["man"])`, plus three of your own. How many work?
3. Compute cosine similarity between "doctor" and "nurse", and between "doctor" with "man" and "woman". What does this suggest about bias in data-driven representations?

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/07-sequences-embeddings.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.